# Private LoRA16 final benchmark evaluation
Use free T4 GPU and Internet. Attach private original Task1 data and pa3-lora16-final-eval-input.zip. No training occurs. Runs full HellaSwag, BoolQ, ARC-Easy and GSM8K with the existing fixed protocol. Outputs contain actual results and generation caches; partial completion is explicitly recorded. Do not overlap another GPU job unless quota and resources are independently confirmed.


In [ ]:
# Set only after the previous Kaggle draft is stopped and no training is active.
BATCH_HANDOFF_CONFIRMED = False
assert BATCH_HANDOFF_CONFIRMED, 'Stop the prior draft/session, then confirm handoff before Save & Run All'


In [ ]:
import pathlib, json
work = pathlib.Path("/kaggle/working")
def restore_verified_inputs(input_root, destination, checkpoint_name, expected, *, match_manifest_only=False):
    import hashlib
    from pathlib import Path, PurePosixPath
    import shutil
    import zipfile

    input_root, destination = Path(input_root), Path(destination)
    if destination.exists():
        raise FileExistsError("Refusing to overwrite an existing recovery directory")
    relative = {}
    for name, digest in expected.items():
        parts = PurePosixPath(name).parts
        if len(parts) < 2 or ".." in parts or PurePosixPath(name).is_absolute():
            raise ValueError("Unsafe checkpoint manifest path")
        relative[str(PurePosixPath(*parts[1:]))] = digest
    marker = checkpoint_name + "/complete.json"
    if marker not in relative:
        raise ValueError("Checkpoint manifest has no complete marker")
    candidates = [("directory", p.parent.parent, "")
                  for p in sorted(input_root.rglob("complete.json"))
                  if p.parent.name == checkpoint_name]
    for archive in sorted(input_root.rglob("*.zip")):
        with zipfile.ZipFile(archive) as source:
            for name in source.namelist():
                if name == marker or name.endswith("/" + marker):
                    candidates.append(("zip", archive, name[:-len(marker)]))
    if not candidates:
        raise ValueError(f"No extracted or zipped complete {checkpoint_name} input found")
    def validate(kind, location, prefix):
        if kind == "directory":
            for name, digest in relative.items():
                file = location / name
                if not file.is_file():
                    raise ValueError(f"Checkpoint file missing: {file}")
                with file.open("rb") as stream:
                    actual = hashlib.file_digest(stream, "sha256").hexdigest()
                if actual != digest:
                    raise ValueError(f"Checkpoint checksum mismatch: {file}")
        else:
            with zipfile.ZipFile(location) as source:
                for name, digest in relative.items():
                    with source.open(prefix + name) as stream:
                        actual = hashlib.file_digest(stream, "sha256").hexdigest()
                    if actual != digest:
                        raise ValueError(f"Checkpoint checksum mismatch: {location}:{prefix + name}")
    # Common names such as step-1250 may belong to other methods. Evaluation
    # may select only exact manifest matches; training keeps strict duplicates.
    verified = []
    for candidate in candidates:
        try:
            validate(*candidate)
        except (ValueError, KeyError, FileNotFoundError):
            if not match_manifest_only:
                raise
        else:
            verified.append(candidate)
    if not verified:
        raise ValueError(f"No {checkpoint_name} input matches the pinned checkpoint manifest")
    kind, location, prefix = verified[0]
    destination.mkdir(parents=True)
    # Copy only pinned files; complete markers come after their recovery payloads.
    order = sorted(relative, key=lambda name: (name.endswith("/complete.json"), name))
    if kind == "directory":
        for name in order:
            target = destination / name
            target.parent.mkdir(parents=True, exist_ok=True)
            with (location / name).open("rb") as source, target.open("xb") as output:
                shutil.copyfileobj(source, output)
    else:
        with zipfile.ZipFile(location) as source:
            for name in order:
                target = destination / name
                target.parent.mkdir(parents=True, exist_ok=True)
                with source.open(prefix + name) as stream, target.open("xb") as output:
                    shutil.copyfileobj(stream, output)
    print("VERIFIED_INPUT_COPIES", len(verified), flush=True)
    return destination / checkpoint_name

EXPECTED = {'lora16-recovery/step-1250/complete.json': '1a72b933ab0ce3640086315afa84306682bbe5768a8e24ef5f7c9a328c58e88d', 'lora16-recovery/step-1250/projected_factors.pt': 'e116df64f1f7b8033e361bd20d307c5f0fa74aa623bfb0fe86cd189b011589c6', 'lora16-recovery/step-1250/summary.json': '5422b9dee42676ecfb7f654abacdbc48200ec9872767b144359d234669b72625', 'lora16-recovery/step-1250/recovery.pt': '94532d5bd7aa5613a1556834c2a31c3aaeadaaae3a57c56ebd9ec81d63adb690'}
checkpoint = restore_verified_inputs('/kaggle/input', work / 'lora16-eval-input', 'step-1250', EXPECTED, match_manifest_only=True)
summary = json.loads((checkpoint / 'summary.json').read_text())
assert summary['update'] == 1250 and summary['successful_optimizer_updates'] == 1250 and summary['skipped_updates'] == 0
print('VERIFIED_FINAL_LORA16_FOR_EVALUATION', summary, flush=True)


In [ ]:
import subprocess, sys, pathlib, json, zipfile, urllib.request
REPO_COMMIT = "cf7f042114a0a924403ed55ef9ee5982946ecf38"
work = pathlib.Path("/kaggle/working")
source = work / "pa3-source"
if not source.exists():
    archive = work / "pa3-source.zip"
    urllib.request.urlretrieve(f"https://github.com/sohauv/atml-pa3/archive/{REPO_COMMIT}.zip", archive)
    with zipfile.ZipFile(archive) as z:
        z.extractall(work)
    (work / f"atml-pa3-{REPO_COMMIT}").rename(source)
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "torch==2.6.0", "torchvision==0.21.0", "torchaudio==2.6.0", "transformers==4.51.3", "accelerate==1.6.0", "datasets==3.5.0", "numpy==2.2.5"])
print("Pinned source prepared:", REPO_COMMIT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'lm_eval==0.4.8', 'peft==0.15.2', 'math-verify[antlr4_13_2]==0.8.0'])


In [ ]:
# Verify all original input bytes; tolerate only byte-identical duplicate copies.
import sys, pathlib, zipfile, hashlib
sys.path.insert(0, str(source))
from scripts.kaggle_task1 import unpack_inputs, select_original_input
EXPECTED_TRAIN_SHA = '644a46f9ab7a8d368499b2054229cc7324614d97596fbdee1ac3ab5cbb584de9'
archives = list(pathlib.Path('/kaggle/input').rglob('pa3-task1-input.zip'))
if len(archives) == 1:
    input_archive = archives[0]
else:
    original = select_original_input('/kaggle/input', EXPECTED_TRAIN_SHA)
    input_archive = work / 'pa3-task1-input.zip'
    if not input_archive.exists():
        with zipfile.ZipFile(input_archive, 'x', zipfile.ZIP_DEFLATED) as z:
            for name in ('train.jsonl', 'metadata.json', 'source_ids.json'):
                z.write(original / name, 'task1/' + name)
data_dir = unpack_inputs(input_archive, work / 'pa3/data')
assert hashlib.file_digest((data_dir / 'train.jsonl').open('rb'), 'sha256').hexdigest() == EXPECTED_TRAIN_SHA
print('Original training data checksum verified:', data_dir, flush=True)


In [ ]:
import subprocess, sys, pathlib, zipfile, hashlib, json
output = work / 'lora16-final-evaluation'
command = [sys.executable, '-m', 'scripts.kaggle_evaluate_task1', '--data-dir', str(data_dir), '--checkpoint', str(checkpoint), '--output', str(output), '--seconds', '18000']
print('START_FULL_LORA16_BENCHMARKS', command, flush=True)
result = subprocess.run(command, cwd=source)
completed = [task for task in ['hellaswag','boolq','arc_easy','gsm8k_cot'] if (output / (task+'.json')).is_file()]
evidence = {'returncode':result.returncode, 'completed_tasks':completed, 'all_four_complete':len(completed)==4, 'checkpoint_update':1250, 'smoke_limit':None}
(work / 'lora16-evaluation-status.json').write_text(json.dumps(evidence,indent=2))
archive = work / 'lora16-final-evaluation-results.zip'
with zipfile.ZipFile(archive, 'x', zipfile.ZIP_DEFLATED) as z:
    if output.exists():
        for p in output.rglob('*'):
            if p.is_file(): z.write(p,str(pathlib.Path('lora16-final-evaluation') / p.relative_to(output)))
    z.write(work / 'lora16-evaluation-status.json','lora16-evaluation-status.json')
with zipfile.ZipFile(archive) as z: assert z.testzip() is None
print('ACTUAL_EVALUATION_STATUS', json.dumps(evidence), flush=True)
print('DOWNLOAD_RESULTS', archive, archive.stat().st_size, hashlib.file_digest(archive.open('rb'),'sha256').hexdigest(), flush=True)
if not evidence['all_four_complete']: print('INCOMPLETE_EVALUATION: preserve output/cache and inspect failure before any continuation', flush=True)
